In [1]:
!pip install pandas numpy scikit-learn joblib

Defaulting to user installation because normal site-packages is not writeable
     ---------------------------------------- 0.0/61.2 kB ? eta -:--:--
     -------------------- ------------------- 30.7/61.2 kB 1.3 MB/s eta 0:00:01
     ---------------------------------------- 61.2/61.2 kB 1.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   - -------------------------------------- 0.4/8.3 MB 11.8 MB/s eta 0:00:01
   --- ------------------------------------ 0.6/8.3 MB 8.0 MB/s eta 0:00:01
   ---- ----------------------------------- 0.8/8.3 MB 7.6 MB/s eta 0:00:01
   ----- ---------------------------------- 1.1/8.3 MB 6.2 MB/s eta 0:00:02
   ------ --------------------------------- 1.2/8.3 MB 5.7 MB/s eta 0:00:02
   ------ --------------------------------- 1.4/8.3 MB 5.2 MB/s eta 0:00:02
   ------- -------------------------------- 1.5/8.3 MB 4.9 MB/s eta 0:00:02
   -------- ------------------------------- 1.8/8.3 MB 4.9 MB/s eta 0:00:02
   ---------


[notice] A new release of pip is available: 24.0 -> 26.2.2
[notice] To update, run: C:\Program Files\Python312\python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

In [3]:
df = pd.read_csv("DeSense_Feature_Engineered_Dataset.csv")

df.head()

,Unnamed: 0,Ride_ID,Rider_ID,Driver_ID,Pickup_Location,Dropoff_Location,City,Ride_Date,Request_Time,Pickup_Time,...,Discount_Amount,Tip_Amount,Driver_Rating,Rider_Rating,Ride_Status,Ride_Year,Ride_Month,Fare_Per_KM,Ride_Status_Category,Ride_Duration_Minutes
0,0,RID100001,R2307,D814,Sector 17,Manimajra,Chandigarh,2026-06-05,07:37,NaN,...,20.0,10.0,5.0,5,Cancelled by Rider,2026.0,6.0,26.629231,Not Completed,NaN
1,1,RID100002,R1375,D777,Dadar,Powai,Mumbai,NaN,07:20,NaN,...,20.0,50.0,3.0,4,No Driver Found,NaN,NaN,26.074419,Not Completed,NaN
2,2,RID100003,R3656,D886,NaN,Cyber City,Gurugram,NaN,03:26,NaN,...,30.0,0.0,5.0,4,Cancelled by Rider,NaN,NaN,18.016667,Not Completed,NaN
3,3,RID100004,R2133,D621,Sarabha Nagar,Model Town,Ludhiana,NaN,13:41,NaN,...,0.0,10.0,5.0,5,No Driver Found,NaN,NaN,16.400787,Not Completed,NaN
4,4,RID100005,R4716,D103,Indiranagar,Whitefield,Bengaluru,NaN,05:13,NaN,...,20.0,0.0,5.0,3,Cancelled by Driver,NaN,NaN,31.262500,Not Completed,NaN


In [4]:
print("Dataset Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nMissing Values:")
print(df.isnull().sum())

Dataset Shape: (10112, 27)

Columns:
['Unnamed: 0', 'Ride_ID', 'Rider_ID', 'Driver_ID', 'Pickup_Location', 'Dropoff_Location', 'City', 'Ride_Date', 'Request_Time', 'Pickup_Time', 'Dropoff_Time', 'Ride_Distance_km', 'Vehicle_Type', 'Ride_Type', 'Payment_Method', 'Fare_Amount', 'Surge_Multiplier', 'Discount_Amount', 'Tip_Amount', 'Driver_Rating', 'Rider_Rating', 'Ride_Status', 'Ride_Year', 'Ride_Month', 'Fare_Per_KM', 'Ride_Status_Category', 'Ride_Duration_Minutes']

Missing Values:
Unnamed: 0                  0
Ride_ID                     0
Rider_ID                    0
Driver_ID                 121
Pickup_Location            40
Dropoff_Location           40
City                        0
Ride_Date                8799
Request_Time                0
Pickup_Time              5076
Dropoff_Time             5076
Ride_Distance_km            0
Vehicle_Type                0
Ride_Type                   0
Payment_Method             30
Fare_Amount                60
Surge_Multiplier            0
Disc

In [5]:
features = [
    "Ride_Distance_km",
    "Vehicle_Type",
    "City",
    "Surge_Multiplier"
]

target = "Fare_Amount"

X = df[features]
y = pd.to_numeric(df[target], errors="coerce")

print("Input Features:")
print(X.head())

print("\nTarget:")
print(y.head())

Input Features:
   Ride_Distance_km Vehicle_Type        City  Surge_Multiplier
0               6.5          SUV  Chandigarh               1.0
1               4.3         Auto      Mumbai               1.2
2               7.2         Auto    Gurugram               1.0
3              12.7         Auto    Ludhiana               1.0
4               4.0        Sedan   Bengaluru               1.2

Target:
0    173.09
1    112.12
2    129.72
3    208.29
4    125.05
Name: Fare_Amount, dtype: float64


In [7]:

data = X.copy()
data["Fare_Amount"] = y

data = data.dropna(subset=["Fare_Amount"])

data = data[data["Ride_Distance_km"] > 0]


X = data[features]
y = data["Fare_Amount"]

print("Final Dataset Shape:", X.shape)

Final Dataset Shape: (10052, 4)


In [8]:
numeric_features = [
    "Ride_Distance_km",
    "Surge_Multiplier"
]

categorical_features = [
    "Vehicle_Type",
    "City"
]

In [9]:
numeric_transformer = SimpleImputer(
    strategy="median"
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training Data:", X_train.shape)
print("Testing Data:", X_test.shape)

Training Data: (8041, 4)
Testing Data: (2011, 4)


In [11]:
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", RandomForestRegressor(
            n_estimators=100,
            random_state=42
        ))
    ]
)

model.fit(X_train, y_train)

print("Model Training Completed!")

Model Training Completed!


In [12]:
y_pred = model.predict(X_test)

print("Actual Fare:")
print(y_test.head())

print("\nPredicted Fare:")
print(y_pred[:5])

Actual Fare:
3173    201.22
2751    197.85
451     342.02
7256    108.09
1814     87.09
Name: Fare_Amount, dtype: float64

Predicted Fare:
[216.35113    213.8204     355.0228     106.23497405  87.34591   ]


In [13]:
mae = mean_absolute_error(y_test, y_pred)

mse = mean_squared_error(y_test, y_pred)

rmse = np.sqrt(mse)

r2 = r2_score(y_test, y_pred)

print("Model Performance")

print("MAE:", mae)
print("MSE:", mse)
print("RMSE:", rmse)
print("R2 Score:", r2)

Model Performance
MAE: 16.13716031109353
MSE: 6981.707861800817
RMSE: 83.55661471003249
R2 Score: 0.9581011005695413


In [14]:
joblib.dump(
    model,
    "../dashboard/fare_prediction_model.pkl"
)

print("Model Saved Successfully!")

Model Saved Successfully!


In [15]:
new_ride = pd.DataFrame({
    "Ride_Distance_km": [12.5],
    "Vehicle_Type": ["Car"],
    "City": ["Ludhiana"],
    "Surge_Multiplier": [1.2]
})

predicted_fare = model.predict(new_ride)

print("Estimated Ride Fare: ₹", round(predicted_fare[0], 2))

Estimated Ride Fare: ₹ 249.1
